---
title: "Chapter 04: K-Nearest Neighbors"
subtitle: "Distance metrics, the KNN algorithm, and choosing K"
date: last-modified
categories: [beginner, classification, regression, distance-metrics]
toc: true
toc-depth: 3
number-sections: false
code-fold: false
code-tools: true
jupyter: python3
---

# Chapter 04: K-Nearest Neighbors (KNN)

> **Level**: Beginner | **Estimated Time**: 2–3 hours

---

## 4.1 Intuition

> "Tell me who your neighbors are, and I'll tell you who you are."

KNN is one of the most intuitive ML algorithms: to classify a new point, look at the **K closest training examples** and take a majority vote.

No training phase — it memorizes the entire dataset and does all computation at prediction time (**lazy learner**).

---

## 4.2 Distance Metrics

### Euclidean Distance (L2)

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Most common. Sensitive to scale → always normalize first!

### Manhattan Distance (L1)

$$d(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{n} |a_i - b_i|$$

Less sensitive to outliers.

### Minkowski Distance (General)

$$d(\mathbf{a}, \mathbf{b}) = \left(\sum_{i=1}^{n} |a_i - b_i|^p\right)^{1/p}$$

- $p=1$: Manhattan
- $p=2$: Euclidean

---

## 4.3 The KNN Algorithm

**Classification:**
1. Compute distance from query point to all training points
2. Sort by distance, select the K nearest neighbors
3. Count votes: majority class wins

**Regression:**
1. Steps 1–2 same
3. Return the **mean** of the K neighbors' target values

### Choosing K
- **K=1**: Very flexible, high variance (overfits, captures noise)
- **K=n**: Very rigid, high bias (just predicts the majority class always)
- **Sweet spot**: Usually odd K (to avoid ties), often tried: 3, 5, 7, 11

As $K$ increases: Variance $\downarrow$, Bias $\uparrow$

As $K$ decreases: Variance $\uparrow$, Bias $\downarrow$

---

## 4.4 From-Scratch Python Implementation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zia207/ML-python/blob/main/Colab_Notebook/chapter-04-knn.ipynb)

In [ ]:
# knn.py
import math

def euclidean_distance(a, b):
    """L2 distance between two vectors."""
    return math.sqrt(sum((ai - bi)**2 for ai, bi in zip(a, b)))

def manhattan_distance(a, b):
    """L1 distance between two vectors."""
    return sum(abs(ai - bi) for ai, bi in zip(a, b))


class KNNClassifier:
    """
    K-Nearest Neighbors Classifier.
    Lazy learner — no training, all computation at prediction time.
    """

    def __init__(self, k=3, distance_fn=euclidean_distance):
        self.k = k
        self.distance_fn = distance_fn
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        """'Training' = simply store the data."""
        self.X_train = X
        self.y_train = y
        return self

    def _predict_one(self, x):
        # Compute distances to all training points
        distances = [
            (self.distance_fn(x, xi), yi)
            for xi, yi in zip(self.X_train, self.y_train)
        ]

        # Sort by distance, pick K nearest
        distances.sort(key=lambda d: d[0])
        k_nearest = distances[:self.k]

        # Majority vote
        vote_counts = {}
        for _, label in k_nearest:
            vote_counts[label] = vote_counts.get(label, 0) + 1

        return max(vote_counts, key=vote_counts.get)

    def predict(self, X):
        return [self._predict_one(xi) for xi in X]

    def predict_proba(self, X):
        """Return vote fractions as class probabilities."""
        results = []
        for xi in X:
            distances = [
                (self.distance_fn(xi, xt), yt)
                for xt, yt in zip(self.X_train, self.y_train)
            ]
            distances.sort(key=lambda d: d[0])
            k_nearest = distances[:self.k]
            labels = [label for _, label in k_nearest]
            classes = sorted(set(self.y_train))
            probs = {c: labels.count(c) / self.k for c in classes}
            results.append(probs)
        return results


class KNNRegressor:
    """K-Nearest Neighbors Regressor."""

    def __init__(self, k=3, distance_fn=euclidean_distance):
        self.k = k
        self.distance_fn = distance_fn
        self.X_train = self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def _predict_one(self, x):
        distances = [
            (self.distance_fn(x, xi), yi)
            for xi, yi in zip(self.X_train, self.y_train)
        ]
        distances.sort(key=lambda d: d[0])
        k_nearest_vals = [yi for _, yi in distances[:self.k]]
        return sum(k_nearest_vals) / len(k_nearest_vals)

    def predict(self, X):
        return [self._predict_one(xi) for xi in X]


# ── K Selection via Cross-Validation ─────────────────────────────────────

def find_best_k(X, y, k_range=range(1, 21), n_folds=5):
    """
    Use k-fold cross-validation to find the best K.
    Returns (best_k, accuracy_per_k).
    """
    n = len(X)
    fold_size = n // n_folds

    results = {}
    for k in k_range:
        fold_accs = []
        for fold in range(n_folds):
            # Split into train and validation
            val_start = fold * fold_size
            val_end   = val_start + fold_size
            X_val = X[val_start:val_end]
            y_val = y[val_start:val_end]
            X_tr  = X[:val_start] + X[val_end:]
            y_tr  = y[:val_start] + y[val_end:]

            clf = KNNClassifier(k=k)
            clf.fit(X_tr, y_tr)
            preds = clf.predict(X_val)
            acc = sum(yp == yt for yp, yt in zip(preds, y_val)) / len(y_val)
            fold_accs.append(acc)

        results[k] = sum(fold_accs) / len(fold_accs)

    best_k = max(results, key=results.get)
    return best_k, results


# ── Demo ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Iris-like toy dataset: [petal_length, petal_width] → species (0,1,2)
    X_train = [
        [1.4, 0.2], [1.3, 0.2], [1.5, 0.1],   # setosa (0)
        [4.7, 1.4], [4.5, 1.5], [4.9, 1.5],   # versicolor (1)
        [6.3, 1.8], [5.8, 1.8], [7.1, 2.1],   # virginica (2)
    ]
    y_train = [0, 0, 0, 1, 1, 1, 2, 2, 2]

    X_test = [[1.6, 0.3], [4.8, 1.6], [6.5, 2.0]]
    y_test = [0, 1, 2]

    for k in [1, 3, 5]:
        clf = KNNClassifier(k=k)
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        acc = sum(yp == yt for yp, yt in zip(preds, y_test)) / len(y_test)
        print(f"K={k}  Predictions: {preds}  Accuracy: {acc:.2f}")

    # Regression demo
    X_reg = [[1.0], [2.0], [3.0], [4.0], [5.0]]
    y_reg = [2.1, 3.9, 6.2, 7.8, 10.1]
    reg = KNNRegressor(k=2)
    reg.fit(X_reg, y_reg)
    print("\nKNN Regression:")
    print(f"  Predict x=2.5 → {reg.predict([[2.5]])[0]:.2f}  (expect ~5.0)")

---

## 4.5 The Curse of Dimensionality

As the number of features (dimensions) grows, KNN degrades dramatically:

- In high dimensions, **all points become approximately equidistant** from each other
- Distance metrics lose meaning
- You need exponentially more data to maintain the same density

**Rule of thumb**: KNN works best with fewer than ~20 features, and always with normalized data.

---

## ✅ Chapter Summary

| Aspect | Detail |
|--------|--------|
| Type | Lazy learner (non-parametric) |
| Training time | $O(1)$ — just store data |
| Prediction time | $O(n \cdot p)$ per query |
| Key hyperparameter | K (number of neighbors) |
| Weakness | Slow at prediction, bad in high dimensions |

---

## 📝 Exercises

1. Implement **weighted KNN** where closer neighbors get higher vote weight ($\text{weight} = \frac{1}{d}$).
2. What happens when K equals the size of the training set?
3. Implement KNN regression with **distance-weighted averaging** instead of simple mean.
4. Add support for the **Cosine similarity** distance metric.

---

**← Previous:** [Chapter 03: Logistic Regression](chapter-03-logistic-regression.qmd)
**→ Next:** [Chapter 05: Naive Bayes](chapter-05-naive-bayes.qmd)